# A02 - 二维 N-S 制造解：Taylor-Green 型验证算例

复杂方程直接上真实问题时，很难判断代码错了还是模型没训好。制造解方法 MMS 的思路是先指定一个解析速度压力场，再反推出方程源项。

这里使用稳态不可压 N-S：

- `u = sin(pi*x)*cos(pi*y)`
- `v = -cos(pi*x)*sin(pi*y)`
- `p = 0.25*(cos(2*pi*x)+cos(2*pi*y))`

该速度场严格满足不可压连续方程。notebook 会用自动微分计算源项，使解析场成为精确解。

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(102)
device = "cuda" if torch.cuda.is_available() else "cpu"
Re = 50.0
device

In [ ]:
class MLP(nn.Module):
    def __init__(self, width=72, depth=5):
        super().__init__()
        layers = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 3)]
        self.net = nn.Sequential(*layers)
    def forward(self, xy):
        return self.net(xy)

def grad(y, x):
    return torch.autograd.grad(y, x, torch.ones_like(y), create_graph=True)[0]

def exact_fields(x, y):
    u = torch.sin(math.pi * x) * torch.cos(math.pi * y)
    v = -torch.cos(math.pi * x) * torch.sin(math.pi * y)
    p = 0.25 * (torch.cos(2 * math.pi * x) + torch.cos(2 * math.pi * y))
    return u, v, p

def exact_source(x, y):
    x.requires_grad_(True)
    y.requires_grad_(True)
    u, v, p = exact_fields(x, y)
    u_x, u_y = grad(u, x), grad(u, y)
    v_x, v_y = grad(v, x), grad(v, y)
    p_x, p_y = grad(p, x), grad(p, y)
    u_xx, u_yy = grad(u_x, x), grad(u_y, y)
    v_xx, v_yy = grad(v_x, x), grad(v_y, y)
    fx = u * u_x + v * u_y + p_x - (u_xx + u_yy) / Re
    fy = u * v_x + v * v_y + p_y - (v_xx + v_yy) / Re
    return fx.detach(), fy.detach()

model = MLP().to(device)

In [ ]:
def residual(x, y):
    fx, fy = exact_source(x.clone(), y.clone())
    x.requires_grad_(True)
    y.requires_grad_(True)
    out = model(torch.cat([x, y], dim=1))
    u, v, p = out[:, 0:1], out[:, 1:2], out[:, 2:3]
    u_x, u_y = grad(u, x), grad(u, y)
    v_x, v_y = grad(v, x), grad(v, y)
    p_x, p_y = grad(p, x), grad(p, y)
    u_xx, u_yy = grad(u_x, x), grad(u_y, y)
    v_xx, v_yy = grad(v_x, x), grad(v_y, y)
    cont = u_x + v_y
    mx = u * u_x + v * u_y + p_x - (u_xx + u_yy) / Re - fx
    my = u * v_x + v * v_y + p_y - (v_xx + v_yy) / Re - fy
    return cont, mx, my

def data_loss(n=256):
    x = torch.rand(n, 1, device=device)
    y = torch.rand(n, 1, device=device)
    u, v, p = exact_fields(x, y)
    pred = model(torch.cat([x, y], dim=1))
    return torch.mean((pred[:,0:1]-u)**2 + (pred[:,1:2]-v)**2 + 0.1*(pred[:,2:3]-p)**2)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
for epoch in range(9000):
    x = torch.rand(1024, 1, device=device)
    y = torch.rand(1024, 1, device=device)
    cont, mx, my = residual(x, y)
    loss_pde = torch.mean(cont**2) + torch.mean(mx**2) + torch.mean(my**2)
    loss_data = data_loss()
    loss = loss_pde + loss_data
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    history.append([loss.item(), loss_pde.item(), loss_data.item()])
    if epoch % 1500 == 0:
        print(f"epoch={epoch:4d}, total={loss.item():.3e}, pde={loss_pde.item():.3e}, data={loss_data.item():.3e}")

In [ ]:
nx = ny = 80
xs = torch.linspace(0, 1, nx, device=device)
ys = torch.linspace(0, 1, ny, device=device)
X, Y = torch.meshgrid(xs, ys, indexing="ij")
xy = torch.cat([X.reshape(-1,1), Y.reshape(-1,1)], dim=1)
with torch.no_grad():
    pred = model(xy)
    U = pred[:,0].reshape(nx, ny)
    V = pred[:,1].reshape(nx, ny)
    P = pred[:,2].reshape(nx, ny)
    Ue, Ve, Pe = exact_fields(X, Y)
    err = torch.sqrt((U-Ue)**2 + (V-Ve)**2)

plt.figure(figsize=(12, 4))
for i, (field, title) in enumerate([(U, "u pred"), (V, "v pred"), (err, "velocity error")], 1):
    plt.subplot(1, 3, i)
    plt.imshow(field.cpu().T, origin="lower", extent=[0,1,0,1])
    plt.colorbar(); plt.title(title)
plt.tight_layout(); plt.show()
print("max velocity error=", torch.max(err).item())